# 05 — Anomaly detection on sensor streams

Two complementary unsupervised detectors trained on the **early-life** (first 25%) cycles of every engine — assumed healthy:

1. **Isolation Forest** on the engineered tabular gold features (notebook 01). Fast, point-wise, and **SHAP-explainable** for governance.
2. **LSTM autoencoder** on raw z-scored sensor windows. Captures *temporal* anomalies a tabular model can't.

**Why this notebook matters:** the LHH-style mining DS JD listed *Anomaly detection* and *predicción de fallas y eventos* as excluyentes. Together with notebooks 02–04 this closes the full set: **RUL regression, survival analysis, anomaly detection.**

**Key metric for predictive maintenance:** *lead time* — how many cycles before failure does the detector first fire a sustained alarm? More lead time = more time to schedule a part order, plan a maintenance window, avoid an unplanned outage.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import time
import warnings
warnings.filterwarnings('ignore')

import mlflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from turboguard.data.cmapss import load_cmapss, add_rul_clipped
from turboguard.features.pipeline import find_constant_sensors, load_gold
from turboguard.models.anomaly.isolation_forest import (
    train_isolation_forest, score_engines, early_warning_lead, shap_top_drivers
)
from turboguard.models.anomaly.autoencoder import train_autoencoder, score_autoencoder

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
GOLD_PATH = ROOT / 'data' / 'processed' / 'gold_FD001.parquet'

mlflow.set_tracking_uri(f'file:{(ROOT / "mlruns").as_posix()}')
mlflow.set_experiment('turboguard-FD001')

## 1. Load gold (engineered features) and raw FD001

In [ ]:
gold = load_gold(GOLD_PATH)
gold = add_rul_clipped(gold, cap=125)
print(f'Gold features: {gold.shape[0]:,} rows × {gold.shape[1]} cols')

fd001 = load_cmapss('FD001')
df_raw = fd001.with_rul()
all_sensors = [c for c in df_raw.columns if c.startswith('sensor_')]
sensor_cols = [c for c in all_sensors if c not in find_constant_sensors(df_raw, all_sensors)]
print(f'Raw sensors: {len(sensor_cols)} non-constant')

## 2. Isolation Forest — fit on first-25% healthy cycles

In [ ]:
t0 = time.time()
if_result = train_isolation_forest(
    gold,
    healthy_fraction=0.25,
    contamination=0.05,
    threshold_quantile=0.01,  # conservative: flag only the bottom 1% of healthy.
    n_estimators=200,
    rng_seed=0,
    run_name='if-FD001',
)
print(f'Trained in {time.time()-t0:.1f}s | features={len(if_result.feature_cols)} | threshold={if_result.threshold:.3f}')

if_scored = score_engines(if_result, gold)
print(f'Overall anomaly rate: {if_scored.is_anomaly.mean():.3f}')

# Anomaly rate by RUL bucket — should rise as engines approach failure.
merged = gold[['unit_id', 'cycle', 'RUL']].merge(if_scored, on=['unit_id', 'cycle'])
buckets = pd.cut(merged['RUL'], bins=[-1, 30, 60, 100, 200, 1000], labels=['0–30', '30–60', '60–100', '100–200', '200+'])
merged.assign(rul_bucket=buckets).groupby('rul_bucket', observed=True)['is_anomaly'].mean().to_frame('anomaly_rate')

## 3. Lead time — first sustained alarm vs. failure cycle

In [ ]:
leads = early_warning_lead(if_scored, gold, consecutive=3)
valid = leads.dropna(subset=['lead_time'])
print(f'Engines with at least one consecutive-3 alarm: {len(valid)}/{len(leads)}')
print(f'Lead time — mean: {valid.lead_time.mean():.1f} cycles | median: {valid.lead_time.median():.1f} cycles')

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.hist(valid['lead_time'], bins=25, edgecolor='black', alpha=0.8)
ax.axvline(valid.lead_time.median(), color='red', linestyle='--', alpha=0.6,
           label=f'median = {valid.lead_time.median():.0f} cycles')
ax.set_xlabel('lead time (cycles before failure)')
ax.set_ylabel('engines')
ax.set_title('Isolation Forest — lead time distribution on FD001')
ax.legend()
fig.tight_layout()

## 4. Explainability — SHAP top drivers of anomaly score

Which features push a cycle into the *anomalous* region? This is the part that makes the detector auditable: every alarm can be traced to specific sensor/feature contributions.

In [ ]:
drivers = shap_top_drivers(if_result, gold, sample_size=300, top_k=15)
drivers

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(drivers['feature'].iloc[::-1], drivers['mean_abs_shap'].iloc[::-1])
ax.set_xlabel('mean |SHAP value|')
ax.set_title('Top 15 features driving the anomaly score (Isolation Forest)')
fig.tight_layout()

## 5. LSTM autoencoder — fit on raw healthy windows

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

t0 = time.time()
ae_result = train_autoencoder(
    df_raw, sensor_cols,
    window=30, hidden_size=32, latent_size=8,
    healthy_fraction=0.25, max_epochs=15, batch_size=128,
    threshold_quantile=0.95,
    device=device, run_name='ae-FD001',
)
print(f'Trained in {time.time()-t0:.1f}s | final loss={ae_result.train_loss:.4f} | threshold={ae_result.threshold:.4f}')

In [ ]:
ae_scored = score_autoencoder(ae_result, df_raw, device=device)
merged_ae = df_raw[['unit_id', 'cycle', 'RUL']].merge(ae_scored, on=['unit_id', 'cycle'])
buckets = pd.cut(merged_ae['RUL'], bins=[-1, 30, 60, 100, 200, 1000], labels=['0–30', '30–60', '60–100', '100–200', '200+'])
summary_ae = merged_ae.assign(rul_bucket=buckets).groupby('rul_bucket', observed=True)['is_anomaly'].mean().to_frame('anomaly_rate')
summary_ae

## 6. Reconstruction-error trajectory for a few engines

Visualise how the autoencoder's reconstruction error grows as an engine degrades — the cleanest single plot in the notebook.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
for uid in [1, 25, 50, 75, 100]:
    sub = ae_scored[ae_scored.unit_id == uid].sort_values('cycle')
    failure_cycle = df_raw[df_raw.unit_id == uid].cycle.max()
    cycles_to_failure = failure_cycle - sub['cycle'].to_numpy()
    ax.plot(cycles_to_failure[::-1], sub['recon_mse'].to_numpy()[::-1], alpha=0.7, label=f'unit {uid}')
ax.axhline(ae_result.threshold, color='red', linestyle='--', alpha=0.6, label=f'threshold = {ae_result.threshold:.3f}')
ax.set_xlabel('cycles to failure (countdown — left = far from failure, right = at failure)')
ax.set_ylabel('reconstruction MSE (per window)')
ax.set_title('LSTM autoencoder — reconstruction error rises as engines approach failure')
ax.invert_xaxis()  # so 'time' flows left-to-right.
ax.legend(loc='best', fontsize=9)
fig.tight_layout()

## 7. Comparison & honest take

| Detector | Input | Strengths | Weaknesses |
|---|---|---|---|
| Isolation Forest | Engineered features | Fast, **SHAP-explainable**, high recall | Higher false-positive rate, no temporal context |
| LSTM autoencoder | Raw sensor windows | Captures **temporal** drifts, sharp early-vs-late ratio | Black-box, no per-feature attribution |

**Production pattern:** the two detectors disagree in different ways, so combining them (e.g. *anomaly = IF flag AND AE flag*, or weighted ensemble) reduces both false-positive and false-negative rates compared to either alone. SHAP attribution from IF supplies the *why* even when the AE is the one firing.

## End of the modeling track

Five notebooks now cover the full LHH-style mining DS JD checklist:

* **00 — EDA** (sanity, baseline)
* **01 — Feature engineering** (rolling, FFT, change-point)
* **02 — RUL regression** (XGBoost, LightGBM)
* **03 — RUL with deep learning** (LSTM)
* **04 — Survival analysis** (Weibull AFT, Cox PH)
* **05 — Anomaly detection** (Isolation Forest + SHAP, LSTM autoencoder) ← this one

Logical next steps for the project: (a) Streamlit demo wiring all five into a single app, (b) Phase 2 — synthetic SAG mill case study to demonstrate transferability to mining heavy equipment.